# Semiconductor exposure

These are executed preliminary results with retrospective explanation; full independent replication has not been established. Historical evaluation blocks were reused; they are not fresh holdouts.

Read the saved outputs without executing this notebook. Original code cells, their metadata, execution counts and outputs are preserved. This package supplies selected scientific evidence and context; the code retains its original source environment and frozen-input dependencies. It is not a standalone reproduction environment. See the [notebook index](README.md).


### Context: direction and variance are different hypotheses

This chapter asks whether signed S or nonnegative V adds useful future-risk information **after** B. A negative S can still produce positive V because V squares the loading and scales by historical residualized-SMH variance. This is a test of variance composition beyond total trailing/recent risk, not a forecast of future SMH shock variance.

Fit B, B+S, B+V and B+S+V separately on their full union-input sample. Do not credit a feature for a change in membership. The primary set scores 26 quarters, 2020Q1–2026Q2, 464–494 issuers per quarter; its actual sample ID matches the main calibration set here. The shorter-window alternative uses a different eligible set. 


## Research question: Signed exposure and the variance component

**Historical evaluation; the periods have been inspected and reused.**

Does semiconductor direction or estimated historical variance exposure add information beyond B? Both positive and negative loadings can transmit variance; a weak signed S result does not settle V.

The saved calculations retain comparison and sample identifiers.


## Direction, variance and their conditional increments

Separately fit B, B+S, B+V and B+S+V on the same union-input sample. Required pairs: B vs B+S, B vs B+V, B+S vs joint, B+V vs joint. Refit B if S/V eligibility reduces coverage; never attribute different-sample score changes to a feature. Keep conventional loadings in C.


In [1]:
from pathlib import Path
import nbformat
ROOT=Path.cwd()
while not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
EXT=ROOT/'extensions/company_risk'
# Execute only tagged definition cells: no earlier notebook scoring or side effects.
shared=nbformat.read(EXT/'notebooks/03_liquidity_recency_calibration.ipynb',4)
for cell in shared.cells:
    if 'shared-company-definitions' in cell.metadata.get('tags',[]): exec(compile(cell.source,'03 shared company definitions','exec'))
features=pd.read_csv(EXT/'data/processed/formation_measurements.csv',parse_dates=['formation_date'])
outcomes=pd.read_csv(EXT/'data/processed/company_outcomes.csv',parse_dates=['formation_date','outcome_end'])
manifest=json.loads((EXT/'specification/comparison_manifest.json').read_text())

runs=[s for s in manifest['runs'] if s['family'] in ['semiconductor','matched_horizon_semiconductor']]
primary=next(s for s in runs if s['run_id']=='semiconductor_main')
runs.append(dict(primary,run_id='jointSV_matchedMixed',models=['B','B+S+V'],pairs=[['B','B+S+V']]))
def fit_semiconductor_comparison_set(features,outcomes,frozen_runs):
    """Fit every predefined model independently with the shared chronological engine."""
    return run_frozen_company_comparisons(features,outcomes,frozen_runs,'semiconductor')
summary=fit_semiconductor_comparison_set(features,outcomes,runs)


## Does composition add information beyond risk level?

V is a historical component of conventional residual variance and can overlap mechanically with long risk and R. Its coefficient in the company forecast differs from daily-return loadings S and A. A useful V forecast increment would show conditional predictive information, not a pure causal shock or forecast of future SMH variance.


In [2]:
display(summary)

,run_id,baseline_model_id,extended_model_id,block_label,quarters,mean_percentage_gain,conditional_low,conditional_high,mean_absolute_gain,mean_calibration_percentage_gain,mean_dispersion_percentage_gain,mean_qlike_gain,positive_quarter_share,calendar_segments,bootstrap_replications,block_length,seed
0,jointSV_matchedL252,B,B+S+V,all,26,-0.112956,-0.514224,0.244878,-0.000309,-0.142844,0.029888,-0.001104,0.500000,1,2000,4,20261014
1,jointSV_matchedL252,B,B+S+V,confirmation,6,-0.493335,-0.865826,-0.131774,-0.000474,-0.333672,-0.159663,-0.000993,0.000000,1,2000,4,20261014
2,jointSV_matchedL252,B,B+S+V,development,12,-0.258769,-0.778210,0.214548,-0.000603,-0.224551,-0.034218,-0.002384,0.500000,1,2000,4,20261014
3,jointSV_matchedL252,B,B+S+V,validation,8,0.391049,0.139306,0.642791,0.000257,0.122839,0.268209,0.000732,0.875000,1,2000,4,20261014
4,jointSV_matchedL60,B,B+S+V,all,26,-0.077044,-0.456478,0.280540,-0.000320,-0.128838,0.051794,-0.001320,0.500000,1,2000,4,20261014
5,jointSV_matchedL60,B,B+S+V,confirmation,6,-0.549576,-0.917484,-0.166616,-0.000519,-0.347776,-0.201800,-0.001133,0.000000,1,2000,4,20261014
6,jointSV_matchedL60,B,B+S+V,development,12,-0.165693,-0.626512,0.297299,-0.000620,-0.206007,0.040314,-0.002809,0.500000,1,2000,4,20261014
7,jointSV_matchedL60,B,B+S+V,validation,8,0.410329,0.137939,0.682719,0.000277,0.151120,0.259209,0.000773,0.875000,1,2000,4,20261014
8,jointSV_matchedMixed,B,B+S+V,all,26,-0.131650,-0.566221,0.256997,-0.000352,-0.170731,0.039080,-0.001238,0.538462,1,2000,4,20261014
9,jointSV_matchedMixed,B,B+S+V,confirmation,6,-0.538963,-0.904572,-0.167023,-0.000513,-0.357454,-0.181509,-0.001099,0.000000,1,2000,4,20261014


### Read the main four pairs separately

| Primary pair | Mean quarterly MSE gain (%) | Conditional 95% interval (%) |
|---|---:|---:|
| B→B+S | −0.095 | [−0.511,0.281] |
| B→B+V | −0.063 | [−0.227,0.107] |
| B+S→B+S+V | −0.037 | [−0.193,0.120] |
| B+V→B+S+V | −0.068 | [−0.459,0.287] |

Source: company_inference.csv, `semiconductor_main`, `block_label=all`. Positive means lower unsmeared log-volatility MSE. All four estimates are unfavorable and intervals span zero. The descriptive common-horizon joint bridge `jointSV_matchedMixed`, B→B+S+V, is **−0.132% [−0.566,0.257]**. Do not add mean percentage increments with different quarterly denominators.

The primary S comparison gains **+0.406%** in validation but **−0.461% [−0.705,−0.177]** in confirmation. V's corresponding means are +0.019% and −0.029%, both uncertain. One favorable block cannot establish stability.


## Bounded robustness without a search grid

Repeat the named four pairs under industry-group controls and separately the 252-session exposure window, one change at a time. Joint B vs B+S+V is the fixed semiconductor matched-liquidity-horizon check, on common samples; retain primary mixed L as the bridge. Do not cross all robustness dimensions.


In [3]:
display(summary[summary.block_label.eq('all')])

,run_id,baseline_model_id,extended_model_id,block_label,quarters,mean_percentage_gain,conditional_low,conditional_high,mean_absolute_gain,mean_calibration_percentage_gain,mean_dispersion_percentage_gain,mean_qlike_gain,positive_quarter_share,calendar_segments,bootstrap_replications,block_length,seed
0,jointSV_matchedL252,B,B+S+V,all,26,-0.112956,-0.514224,0.244878,-0.000309,-0.142844,0.029888,-0.001104,0.500000,1,2000,4,20261014
4,jointSV_matchedL60,B,B+S+V,all,26,-0.077044,-0.456478,0.280540,-0.000320,-0.128838,0.051794,-0.001320,0.500000,1,2000,4,20261014
8,jointSV_matchedMixed,B,B+S+V,all,26,-0.131650,-0.566221,0.256997,-0.000352,-0.170731,0.039080,-0.001238,0.538462,1,2000,4,20261014
12,semiconductor_industry_group,B,B+S,all,26,-0.052506,-0.201100,0.073350,-0.000049,-0.042479,-0.010027,-0.000078,0.346154,1,2000,4,20261014
16,semiconductor_industry_group,B,B+V,all,26,-0.072494,-0.168909,0.024541,-0.000193,-0.050926,-0.021568,-0.001063,0.461538,1,2000,4,20261014
20,semiconductor_industry_group,B+S,B+S+V,all,26,-0.050801,-0.211495,0.098982,-0.000213,-0.111993,0.061193,-0.001212,0.576923,1,2000,4,20261014
24,semiconductor_industry_group,B+V,B+S+V,all,26,-0.030833,-0.191121,0.133231,-0.000069,-0.103804,0.072971,-0.000226,0.423077,1,2000,4,20261014
28,semiconductor_main,B,B+S,all,26,-0.094897,-0.510945,0.281364,-0.000184,-0.046840,-0.048057,-0.000229,0.500000,1,2000,4,20261014
32,semiconductor_main,B,B+V,all,26,-0.062678,-0.227341,0.106839,-0.000083,0.004678,-0.067355,-0.000440,0.538462,1,2000,4,20261014
36,semiconductor_main,B+S,B+S+V,all,26,-0.037069,-0.192525,0.120053,-0.000168,-0.124002,0.086933,-0.001008,0.423077,1,2000,4,20261014


### Bounded alternatives: keep favorable and unfavorable evidence

![Semiconductor and asymmetry sensitivity intervals](assets/semiconductor-exposure-semiconductor_sensitivities.png)

The first two panels show all four S/V pairs at main 504-session and shorter 252-session settings; the third previews A. Whiskers are saved conditional intervals, not full-pipeline uncertainty. The horizontal intervals are conditional on the saved fitted inputs and forecasts.

With industry-group rather than sector controls, the four all-period gains are **−0.053%, −0.072%, −0.051%, −0.031%**, all intervals spanning zero. With 252-session exposure measurements they are **−0.094%, +0.133%, +0.005%, −0.222%**, in the same pair order as the table above. The last interval is [−0.463,−0.018]; the favorable V-only estimate spans zero. Shorter-window sets use sample ID `40ed4a5850fad159` and 466–496 issuers per quarter, so main-versus-shorter differences also reflect eligibility.

Fixed horizon joint S/V bridges are −0.132% mixed, −0.077% 60/60 and −0.113% 252/252, all intervals crossing zero. All block/sensitivity rows retain the declared comparisons without a model/horizon search.


## Findings and interpretation

Mean quarterly MSE improvements beyond B are −0.095% for S and −0.063% for V, with conditional intervals [−0.511%, 0.281%] and [−0.227%, 0.107%]. The common-sample joint increment is −0.132%. These small, unstable effects and bounded alternatives do not establish reliable additional semiconductor forecasting value. V represents historical variance composition rather than future shock variance. Negative results and all block labels are retained.


## Related calculations

[Recent risk and liquidity](recent-risk-and-liquidity.ipynb) establishes the benchmark. The supporting [downside-asymmetry calculation](../reproduction/company-risk/extensions/company_risk/notebooks/05_downside_asymmetry.ipynb) asks whether downside exposure adds information after signed exposure and variance composition. Use the reproduction runner to execute that source notebook.


In [4]:
display(summary)

,run_id,baseline_model_id,extended_model_id,block_label,quarters,mean_percentage_gain,conditional_low,conditional_high,mean_absolute_gain,mean_calibration_percentage_gain,mean_dispersion_percentage_gain,mean_qlike_gain,positive_quarter_share,calendar_segments,bootstrap_replications,block_length,seed
0,jointSV_matchedL252,B,B+S+V,all,26,-0.112956,-0.514224,0.244878,-0.000309,-0.142844,0.029888,-0.001104,0.500000,1,2000,4,20261014
1,jointSV_matchedL252,B,B+S+V,confirmation,6,-0.493335,-0.865826,-0.131774,-0.000474,-0.333672,-0.159663,-0.000993,0.000000,1,2000,4,20261014
2,jointSV_matchedL252,B,B+S+V,development,12,-0.258769,-0.778210,0.214548,-0.000603,-0.224551,-0.034218,-0.002384,0.500000,1,2000,4,20261014
3,jointSV_matchedL252,B,B+S+V,validation,8,0.391049,0.139306,0.642791,0.000257,0.122839,0.268209,0.000732,0.875000,1,2000,4,20261014
4,jointSV_matchedL60,B,B+S+V,all,26,-0.077044,-0.456478,0.280540,-0.000320,-0.128838,0.051794,-0.001320,0.500000,1,2000,4,20261014
5,jointSV_matchedL60,B,B+S+V,confirmation,6,-0.549576,-0.917484,-0.166616,-0.000519,-0.347776,-0.201800,-0.001133,0.000000,1,2000,4,20261014
6,jointSV_matchedL60,B,B+S+V,development,12,-0.165693,-0.626512,0.297299,-0.000620,-0.206007,0.040314,-0.002809,0.500000,1,2000,4,20261014
7,jointSV_matchedL60,B,B+S+V,validation,8,0.410329,0.137939,0.682719,0.000277,0.151120,0.259209,0.000773,0.875000,1,2000,4,20261014
8,jointSV_matchedMixed,B,B+S+V,all,26,-0.131650,-0.566221,0.256997,-0.000352,-0.170731,0.039080,-0.001238,0.538462,1,2000,4,20261014
9,jointSV_matchedMixed,B,B+S+V,confirmation,6,-0.538963,-0.904572,-0.167023,-0.000513,-0.357454,-0.181509,-0.001099,0.000000,1,2000,4,20261014


### Takeaway and next question

These saved comparisons do not establish stable added semiconductor forecasting value after B. They do not deny historical semiconductor-related variation: V is partly a decomposition of risk already represented by long risk/R. A null or adverse increment means that identifying this component has not demonstrated extra predictive usefulness here. The supporting downside-asymmetry comparison tests A after S and V, refitting the baseline on the sample eligible for A.
